# Midterm-Style Submission Notebook

This is the only supported submission notebook.
It mirrors the midterm baseline from `DL_Midterm_Eval.ipynb`, but uses the current Google notebook setup for Drive mount, repo checkout, merged-model discovery, and persistent output files.

The active flow keeps the baseline SVG cleanup logic, then runs three separate saved passes:
1. plan objects and layout
2. generate SVG from the prompt plus plan when available
3. revise only weak rows and write the final submission

Historical experiment notebooks are preserved in `archive/` for documentation, but they are not supported execution paths.

This next cell mounts Google Drive, clones the repo into the runtime, copies the required CSV inputs into the checkout, and defines the persistent Drive output directory.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output folders
- Rerun-safe: Yes. It resets the runtime checkout and recopies the inputs.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT_ON_DRIVE = DRIVE_ROOT / "svg-kaggle-comptetition"
OUTPUT_ROOT = PROJECT_ROOT_ON_DRIVE / "submission_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    candidate_paths = [
        PROJECT_ROOT_ON_DRIVE / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            break
    else:
        raise FileNotFoundError(f"Could not find {required_name} in Google Drive.")

os.chdir(CHECKOUT_PATH)
print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Persistent output root: {OUTPUT_ROOT}")


## Package Check

This next cell installs the runtime packages needed by the midterm-style notebook, using normal Python subprocess calls instead of notebook shell magics.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs packages that are missing.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "trl": "trl",
    "cairosvg": "cairosvg",
    "lxml": "lxml",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Imports And Config

This next cell imports the notebook dependencies and defines the same core settings used by the midterm notebook, plus the Drive-backed output paths for this setup.

- Reads: Runtime packages, repo paths
- Writes: In-memory constants and config only
- Rerun-safe: Yes. It only defines notebook state.


In [ ]:
import io
import re
import shutil
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from lxml import etree
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SVG_NS = "http://www.w3.org/2000/svg"
ADAPTER_PATH = CHECKOUT_PATH / "svg-lora-adapter"
MERGED_PATH = CHECKOUT_PATH / "svg-model-merged"
DRIVE_PROJECT_CANDIDATES = [
    PROJECT_ROOT_ON_DRIVE,
    DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition",
]
DRIVE_ADAPTER_PATHS = [root / "svg-lora-adapter" for root in DRIVE_PROJECT_CANDIDATES]
DRIVE_MERGED_PATHS = [root / "svg-model-merged" for root in DRIVE_PROJECT_CANDIDATES]
MERGED_WEIGHT_FILES = (
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
    "pytorch_model.bin.index.json",
)
ADAPTER_REQUIRED_FILES = (
    "adapter_config.json",
    "adapter_model.safetensors",
)


def has_model_weights(path: Path) -> bool:
    return path.exists() and any((path / name).exists() for name in MERGED_WEIGHT_FILES)


def has_adapter_files(path: Path) -> bool:
    return path.exists() and all((path / name).exists() for name in ADAPTER_REQUIRED_FILES)


def first_matching_path(paths, predicate):
    for candidate in paths:
        if predicate(candidate):
            return candidate
    return None


TEST_CSV = CHECKOUT_PATH / "test.csv"
PASS1_PLANS_CSV = OUTPUT_ROOT / "submission_pass1_plans.csv"
PASS2_SUBMISSION_CSV = OUTPUT_ROOT / "submission_pass2.csv"
PASS2_DEBUG_CSV = OUTPUT_ROOT / "submission_pass2_debug.csv"
SUBMISSION_CSV = OUTPUT_ROOT / "submission.csv"
DEBUG_CSV = OUTPUT_ROOT / "submission_debug.csv"

USE_LAYOUT_PLAN = True
USE_REVISE_PASS = True
PLAN_MAX_NEW_TOKENS = 128
PLAN_BATCH_SIZE = 512
SVG_MAX_NEW_TOKENS = 1536
SVG_BATCH_SIZE = 200
REVISE_MAX_NEW_TOKENS = 1792
REVISE_BATCH_SIZE = 100
REVISE_WEAK_REASONS = {"xml", "repaired", "fallback"}

RUN_LIMIT = None  # Set to 10 for a quick smoke test in Colab.

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256
STRICT_VIEWBOX = f"0 0 {RENDER_SIZE} {RENDER_SIZE}"
MIN_REPAIRED_SVG_LENGTH = 80

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    f'<svg xmlns="{SVG_NS}" viewBox="{STRICT_VIEWBOX}" width="{RENDER_SIZE}" height="{RENDER_SIZE}">'
    f'<rect width="{RENDER_SIZE}" height="{RENDER_SIZE}" fill="white"/>'
    '</svg>'
)

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print(f"Test CSV: {TEST_CSV}")
print(f"Pass 1 plans: {PASS1_PLANS_CSV}")
print(f"Pass 2 submission: {PASS2_SUBMISSION_CSV}")
print(f"Pass 2 debug: {PASS2_DEBUG_CSV}")
print(f"Final submission: {SUBMISSION_CSV}")
print(f"Final debug: {DEBUG_CSV}")
print(f"Run limit: {RUN_LIMIT}")
print(f"Use layout plan: {USE_LAYOUT_PLAN}")
print(f"Use revise pass: {USE_REVISE_PASS}")


## Load Model

This next cell follows the midterm-style inference path. It prefers merged model weights, copies them from Drive into the runtime checkout if needed, and only falls back to base model plus LoRA if merged weights are still unavailable.

- Reads: Local merged model directory, Drive merged model directory, adapter files, Hugging Face model files
- Writes: In-memory tokenizer and model state only
- Rerun-safe: Yes. It reloads the model path for the current runtime.


In [ ]:
# TODO: Remove this merged-model autodetect fallback once svg-model-merged has been uploaded to Drive.
local_merged_ready = has_model_weights(MERGED_PATH)
local_adapter_ready = has_adapter_files(ADAPTER_PATH)
drive_merged_path = first_matching_path(DRIVE_MERGED_PATHS, has_model_weights)
drive_adapter_path = first_matching_path(DRIVE_ADAPTER_PATHS, has_adapter_files)

if not local_merged_ready and drive_merged_path is not None:
    if MERGED_PATH.exists():
        shutil.rmtree(MERGED_PATH)
    shutil.copytree(drive_merged_path, MERGED_PATH)
    local_merged_ready = True
    print(f"Copied merged model from Drive: {drive_merged_path}")
elif MERGED_PATH.exists() and not local_merged_ready:
    print(f"Merged model directory exists at {MERGED_PATH}, but no weight files were found.")

if not local_adapter_ready and drive_adapter_path is not None:
    if ADAPTER_PATH.exists():
        shutil.rmtree(ADAPTER_PATH)
    shutil.copytree(drive_adapter_path, ADAPTER_PATH)
    local_adapter_ready = True
    print(f"Copied adapter from Drive: {drive_adapter_path}")
elif ADAPTER_PATH.exists() and not local_adapter_ready:
    print(f"Adapter directory exists at {ADAPTER_PATH}, but required files were not found.")

if local_merged_ready:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MERGED_PATH,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    MODEL_SOURCE = f"merged model at {MERGED_PATH}"
else:
    if not local_adapter_ready:
        raise FileNotFoundError(
            "Merged model weights were not found, and the LoRA adapter is also unavailable. "
            f"Checked merged paths: {MERGED_PATH} plus {DRIVE_MERGED_PATHS}. "
            f"Checked adapter paths: {ADAPTER_PATH} plus {DRIVE_ADAPTER_PATHS}."
        )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH), local_files_only=True)
    MODEL_SOURCE = f"base model + LoRA adapter at {ADAPTER_PATH}"
    print("Merged model not found. Using the temporary base-plus-LoRA fallback.")
    print("Remove this fallback path once svg-model-merged is uploaded to Drive.")

model.eval()
if model.config.pad_token_id is None and tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

for attr_name in ("temperature", "top_p", "top_k"):
    if hasattr(model.generation_config, attr_name):
        setattr(model.generation_config, attr_name, None)

print(f"Loaded model source: {MODEL_SOURCE}")
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


## SVG Helpers

This next cell ports the permissive midterm SVG extraction, repair, XML recovery, and validation helpers.

- Reads: Raw model text, SVG strings
- Writes: In-memory helper functions only
- Rerun-safe: Yes. It only defines functions.


In [ ]:
EVENT_HANDLER_RE = re.compile(r"\son[a-zA-Z]+\s*=", re.IGNORECASE)
EXTERNAL_HREF_RE = re.compile(r"\s(?:href|xlink:href)\s*=\s*[\"']\s*(?:https?:|//)", re.IGNORECASE)
REMOTE_REF_RE = re.compile(r"@import\b|url\(\s*[\"']?(?:https?:|//)", re.IGNORECASE)
ROOT_TAG_RE = re.compile(r"<svg\b[^>]*>", flags=re.IGNORECASE | re.DOTALL)


def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def extract_svg(text: str) -> str:
    text = str(text or "").strip()

    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()

    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()

    return text


def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(
            bytestring=svg.encode("utf-8"),
            output_width=size,
            output_height=size,
        )
        img = Image.open(io.BytesIO(png)).convert("RGB")
        return np.array(img)
    except Exception:
        return None


def get_attr_value(opening_tag: str, attr_name: str):
    pattern = rf"(\s{re.escape(attr_name)}\s*=\s*)([\"'])(.*?)\2"
    match = re.search(pattern, opening_tag, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        return None
    return match.group(3)


def upsert_attr(opening_tag: str, attr_name: str, attr_value: str) -> str:
    pattern = rf"(\s{re.escape(attr_name)}\s*=\s*)([\"'])(.*?)\2"

    def replacer(match: re.Match[str]) -> str:
        prefix = match.group(1)
        quote = match.group(2)
        return f"{prefix}{quote}{attr_value}{quote}"

    if re.search(pattern, opening_tag, flags=re.IGNORECASE | re.DOTALL):
        return re.sub(
            pattern,
            replacer,
            opening_tag,
            count=1,
            flags=re.IGNORECASE | re.DOTALL,
        )
    return opening_tag[:-1] + f' {attr_name}="{attr_value}">'


def normalize_root_attributes(svg: str) -> str:
    match = ROOT_TAG_RE.search(svg)
    if match is None:
        return svg

    opening_tag = match.group(0)
    normalized_tag = opening_tag
    normalized_tag = upsert_attr(normalized_tag, "xmlns", SVG_NS)
    normalized_tag = upsert_attr(normalized_tag, "viewBox", STRICT_VIEWBOX)
    normalized_tag = upsert_attr(normalized_tag, "width", str(RENDER_SIZE))
    normalized_tag = upsert_attr(normalized_tag, "height", str(RENDER_SIZE))
    return svg.replace(opening_tag, normalized_tag, 1)


def strict_contract_issues(svg: str) -> list[str]:
    issues: list[str] = []
    opening_match = ROOT_TAG_RE.search(svg)
    if opening_match is None:
        issues.append("strict_parse_failed")
        return issues

    opening_tag = opening_match.group(0)
    if get_attr_value(opening_tag, "xmlns") != SVG_NS:
        issues.append("missing_xmlns")

    try:
        root = ET.fromstring(svg)
    except Exception:
        issues.append("strict_parse_failed")
        return issues

    if strip_namespace(root.tag) != "svg":
        issues.append("root_not_svg")
        return issues

    if root.attrib.get("viewBox") != STRICT_VIEWBOX:
        issues.append("viewbox_not_exact")

    if root.attrib.get("width") != str(RENDER_SIZE) or root.attrib.get("height") != str(RENDER_SIZE):
        issues.append("width_height_not_exact")

    return issues


def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"

    svg = svg.strip()

    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"

    if EVENT_HANDLER_RE.search(svg):
        return 0, "disallowed_attr:event_handler"

    if EXTERNAL_HREF_RE.search(svg):
        return 0, "disallowed_ref:external_href"

    if REMOTE_REF_RE.search(svg):
        return 0, "disallowed_ref:remote_url"

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)

        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"

        if tag == "path":
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"

    if render_svg(svg) is None:
        return 0, "render_failed"

    return 1, "valid"


def repair_svg(svg: str) -> str:
    if not svg:
        return svg

    svg = svg.strip()

    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]

    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()

    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[: end + len("</svg>")]

    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"

    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)
    svg = re.sub(r"<[^>]*$", "", svg)
    return svg


def recover_svg_with_lxml(svg: str) -> str:
    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg


def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True

    paths = [elem for elem in root.iter() if strip_namespace(elem.tag) == "path"]
    nonempty_paths = [path for path in paths if path.attrib.get("d", "").strip()]

    if len(paths) > 0 and len(nonempty_paths) == 0:
        return True

    total_elems = sum(1 for _ in root.iter())
    if total_elems <= 1:
        return True

    return False


def summarize_stage_failure(source_gate_reason: str, normalized_gate_reason: str) -> str:
    parts = []
    if source_gate_reason and source_gate_reason != "valid":
        parts.append(f"source:{source_gate_reason}")
    if normalized_gate_reason and normalized_gate_reason != "valid" and normalized_gate_reason != source_gate_reason:
        parts.append(f"normalized:{normalized_gate_reason}")
    if parts:
        return ";".join(parts)
    return source_gate_reason or normalized_gate_reason or "unknown_failure"


def candidate_from_svg(raw_text: str, extracted_svg: str, svg: str, reason: str):
    source_valid, source_gate_reason = validity_gate(svg)
    normalized_svg = normalize_root_attributes(svg)
    normalized_valid, normalized_gate_reason = validity_gate(normalized_svg)

    normalization_status = "unchanged"
    final_svg = svg
    gate_reason = source_gate_reason

    if normalized_svg != svg:
        if normalized_valid == 1:
            normalization_status = "applied"
            final_svg = normalized_svg
            gate_reason = normalized_gate_reason
        elif source_valid == 1:
            normalization_status = "reverted_after_failed_normalize"
        else:
            normalization_status = "failed"
    elif normalized_valid == 1:
        gate_reason = normalized_gate_reason

    if source_valid == 0 and normalized_valid == 0:
        return None, summarize_stage_failure(source_gate_reason, normalized_gate_reason)

    collapsed = looks_collapsed(final_svg)
    if reason in {"repaired", "xml"} and (collapsed or len(final_svg) < MIN_REPAIRED_SVG_LENGTH):
        return None, "collapsed_or_too_short"

    strict_issues = strict_contract_issues(final_svg)
    candidate = {
        "reason": reason,
        "gate_reason": gate_reason,
        "source_gate_reason": source_gate_reason,
        "normalized_gate_reason": normalized_gate_reason,
        "normalization_status": normalization_status,
        "strict_contract": not strict_issues,
        "strict_issues": ",".join(strict_issues),
        "raw_text": raw_text,
        "extracted_svg": extracted_svg,
        "normalized_svg": normalized_svg,
        "final_svg": final_svg,
        "generated_tokens": 0,
        "hit_token_cap": False,
        "collapsed": collapsed,
        "raw_gate_reason": "",
        "repaired_gate_reason": "",
        "xml_gate_reason": "",
        "failure_reasons": "",
    }
    return candidate, gate_reason


def clean_svg_output(raw_text: str):
    cleaned_raw_text = str(raw_text or "")
    extracted_svg = extract_svg(cleaned_raw_text)

    valid_candidate, raw_gate_reason = candidate_from_svg(cleaned_raw_text, extracted_svg, extracted_svg, "valid")
    if valid_candidate is not None:
        valid_candidate["raw_gate_reason"] = raw_gate_reason
        valid_candidate["repaired_gate_reason"] = raw_gate_reason
        valid_candidate["xml_gate_reason"] = raw_gate_reason
        return valid_candidate

    repaired_svg = repair_svg(extracted_svg)
    repaired_candidate, repaired_gate_reason = candidate_from_svg(cleaned_raw_text, extracted_svg, repaired_svg, "repaired")
    if repaired_candidate is not None:
        repaired_candidate["raw_gate_reason"] = raw_gate_reason
        repaired_candidate["repaired_gate_reason"] = repaired_gate_reason
        repaired_candidate["xml_gate_reason"] = repaired_gate_reason
        repaired_candidate["failure_reasons"] = f"raw={raw_gate_reason}"
        return repaired_candidate

    xml_svg = recover_svg_with_lxml(repaired_svg)
    xml_candidate, xml_gate_reason = candidate_from_svg(cleaned_raw_text, extracted_svg, xml_svg, "xml")
    if xml_candidate is not None:
        xml_candidate["raw_gate_reason"] = raw_gate_reason
        xml_candidate["repaired_gate_reason"] = repaired_gate_reason
        xml_candidate["xml_gate_reason"] = xml_gate_reason
        xml_candidate["failure_reasons"] = f"raw={raw_gate_reason}|repaired={repaired_gate_reason}"
        return xml_candidate

    fallback_issues = strict_contract_issues(FALLBACK_SVG)
    fallback_gate_reason = xml_gate_reason or repaired_gate_reason or raw_gate_reason or "fallback"
    return {
        "reason": "fallback",
        "gate_reason": fallback_gate_reason,
        "source_gate_reason": "",
        "normalized_gate_reason": "",
        "normalization_status": "fallback",
        "strict_contract": not fallback_issues,
        "strict_issues": ",".join(fallback_issues),
        "raw_text": cleaned_raw_text,
        "extracted_svg": extracted_svg,
        "normalized_svg": FALLBACK_SVG,
        "final_svg": FALLBACK_SVG,
        "generated_tokens": 0,
        "hit_token_cap": False,
        "collapsed": False,
        "raw_gate_reason": raw_gate_reason,
        "repaired_gate_reason": repaired_gate_reason,
        "xml_gate_reason": xml_gate_reason,
        "failure_reasons": f"raw={raw_gate_reason}|repaired={repaired_gate_reason}|xml={xml_gate_reason}",
    }


## Pass Helpers

This next cell defines the lightweight planning, SVG generation, targeted revision, and CSV save helpers used by the three execution passes.

- Reads: Test prompts, tokenizer, model, SVG helper functions
- Writes: In-memory helper functions only
- Rerun-safe: Yes. It only defines functions.


In [ ]:
REASON_RANK = {
    "fallback": 0,
    "xml": 1,
    "repaired": 2,
    "valid": 3,
}

CANDIDATE_COLUMNS = [
    "candidate_source",
    "reason",
    "gate_reason",
    "source_gate_reason",
    "normalized_gate_reason",
    "normalization_status",
    "strict_contract",
    "strict_issues",
    "raw_gate_reason",
    "repaired_gate_reason",
    "xml_gate_reason",
    "failure_reasons",
    "generated_tokens",
    "hit_token_cap",
    "collapsed",
    "raw_text",
    "extracted_svg",
    "normalized_svg",
    "final_svg",
]

CANDIDATE_DEFAULTS = {
    "candidate_source": "",
    "reason": "",
    "gate_reason": "",
    "source_gate_reason": "",
    "normalized_gate_reason": "",
    "normalization_status": "",
    "strict_contract": False,
    "strict_issues": "",
    "raw_gate_reason": "",
    "repaired_gate_reason": "",
    "xml_gate_reason": "",
    "failure_reasons": "",
    "generated_tokens": 0,
    "hit_token_cap": False,
    "collapsed": False,
    "raw_text": "",
    "extracted_svg": "",
    "normalized_svg": "",
    "final_svg": "",
}

CANDIDATE_BOOL_FIELDS = {"strict_contract", "hit_token_cap", "collapsed"}
CANDIDATE_INT_FIELDS = {"generated_tokens"}
PASS1_COLUMNS = [
    "id",
    "prompt",
    "plan_text",
    "plan_source",
    "plan_raw_text",
    "plan_generated_tokens",
    "plan_hit_token_cap",
]

FALLBACK_STRICT_ISSUES = strict_contract_issues(FALLBACK_SVG)
if FALLBACK_STRICT_ISSUES:
    raise ValueError(f"Fallback SVG is not strict-contract compliant: {FALLBACK_STRICT_ISSUES}")


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value != 0
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def as_text(value) -> str:
    if pd.isna(value):
        return ""
    return str(value)


def reorder_columns(df: pd.DataFrame, preferred: list[str]) -> pd.DataFrame:
    columns = [column for column in preferred if column in df.columns]
    columns.extend(column for column in df.columns if column not in columns)
    return df[columns]


def candidate_base_row(candidate=None) -> dict[str, object]:
    if candidate is None:
        return dict(CANDIDATE_DEFAULTS)
    return {column: candidate.get(column, CANDIDATE_DEFAULTS[column]) for column in CANDIDATE_COLUMNS}


def candidate_to_prefixed_row(prefix: str, candidate=None) -> dict[str, object]:
    base_row = candidate_base_row(candidate)
    return {f"{prefix}_{column}": value for column, value in base_row.items()}


def candidate_from_row(row, prefix: str = "") -> dict[str, object]:
    candidate = {}
    for column, default in CANDIDATE_DEFAULTS.items():
        key = f"{prefix}_{column}" if prefix else column
        value = row[key] if key in row else default
        if pd.isna(value):
            value = default
        if column in CANDIDATE_BOOL_FIELDS:
            value = as_bool(value)
        elif column in CANDIDATE_INT_FIELDS:
            value = int(value)
        else:
            value = as_text(value)
        candidate[column] = value
    return candidate


def load_test_df(test_csv_path=TEST_CSV, limit=RUN_LIMIT):
    test_df = pd.read_csv(test_csv_path)
    test_df = test_df.dropna(subset=["id", "prompt"]).copy()
    test_df["prompt"] = test_df["prompt"].astype(str)
    if limit is not None:
        test_df = test_df.head(limit).copy()
    return test_df.reset_index(drop=True)


def save_dataframe(df: pd.DataFrame, path, label: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved {label} to: {path}")


def build_plan_input(prompt: str) -> str:
    return (
        f"Prompt: {prompt}\n"
        "Return only a compact visual plan. Do not output SVG, XML, markdown, or extra commentary.\n"
        "Write exactly four lines, using these headers once each and in this order. Make every line specific to the prompt. Stop after the fourth line.\n"
        "OBJECTS:\n"
        "LAYOUT:\n"
        "PALETTE:\n"
        "SVG_HINTS:\n"
        "Plan:\n"
    )


COLOR_WORDS = [
    "black", "white", "gray", "grey", "red", "orange", "yellow", "green", "blue", "purple",
    "pink", "brown", "gold", "silver", "cyan", "magenta", "teal", "navy", "beige",
]


def shorten_prompt_text(prompt: str, max_words: int = 14) -> str:
    words = re.findall(r"[A-Za-z0-9%+#'-]+", str(prompt))
    if not words:
        return "main subject from the prompt"
    return " ".join(words[:max_words])


def infer_layout_from_prompt(prompt: str) -> str:
    lowered = str(prompt).lower()
    placements = []
    for keyword in ("center", "centered", "left", "right", "top", "bottom", "corner"):
        if keyword in lowered:
            placements.append(keyword)
    if placements:
        return f"emphasize {', '.join(dict.fromkeys(placements))} placement from the prompt"
    return "center the main subject and keep the composition simple"


def infer_palette_from_prompt(prompt: str) -> str:
    lowered = str(prompt).lower()
    colors = [color for color in COLOR_WORDS if re.search(rf"\b{re.escape(color)}\b", lowered)]
    if colors:
        return ", ".join(dict.fromkeys(colors[:4]))
    if any(word in lowered for word in ("outline", "icon", "logo", "silhouette", "line art")):
        return "high contrast monochrome"
    return "prompt-implied colors with restrained contrast"


def infer_svg_hints_from_prompt(prompt: str) -> str:
    lowered = str(prompt).lower()
    hints = []
    if any(word in lowered for word in ("icon", "logo", "symbol", "emblem")):
        hints.append("use simple geometric shapes")
    if any(word in lowered for word in ("outline", "line", "stroke", "sketch")):
        hints.append("prefer clean strokes over filled detail")
    if any(word in lowered for word in ("silhouette", "shadow")):
        hints.append("favor bold filled shapes")
    if any(word in lowered for word in ("minimal", "simple", "clean")):
        hints.append("keep path count low")
    if not hints:
        hints.append("use simple shapes and a modest number of paths")
    return "; ".join(hints[:3])


def build_fallback_plan(prompt: str) -> str:
    return "\n".join([
        f"OBJECTS: {shorten_prompt_text(prompt)}",
        f"LAYOUT: {infer_layout_from_prompt(prompt)}",
        f"PALETTE: {infer_palette_from_prompt(prompt)}",
        f"SVG_HINTS: {infer_svg_hints_from_prompt(prompt)}",
    ])


def clean_plan_output(raw_text: str):
    text = str(raw_text or "").strip().replace("\r", "")
    if not text:
        return None

    headers = ("OBJECTS:", "LAYOUT:", "PALETTE:", "SVG_HINTS:")
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)
    text = text.replace("Plan:\n", " ").replace("Plan:", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return None

    svg_start = text.lower().find("<svg")
    if svg_start != -1:
        text = text[:svg_start]

    escaped_headers = [re.escape(header) for header in headers]
    header_pattern = re.compile(r"(" + "|".join(escaped_headers) + r")", flags=re.IGNORECASE)
    matches = list(header_pattern.finditer(text))
    if len(matches) < 4:
        return None

    extracted = {}
    for idx, match in enumerate(matches):
        header = match.group(1).upper()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        if header in extracted:
            continue
        content = text[start:end].strip(" :-;,.\n\t")
        if not content:
            continue
        chunks = [chunk.strip(" :-") for chunk in re.split(r"[;|]", content) if chunk.strip()]
        filtered_chunks = []
        for chunk in chunks:
            lowered = chunk.lower()
            if any(
                phrase in lowered
                for phrase in (
                    "return only a compact visual plan",
                    "do not output svg",
                    "write exactly four lines",
                    "using these headers",
                    "stop after the fourth line",
                    "now write the plan",
                    "using the same four headers only",
                    "example",
                )
            ):
                continue
            filtered_chunks.append(chunk)
        cleaned_content = "; ".join(filtered_chunks).strip()
        cleaned_content = re.sub(r"\s+", " ", cleaned_content)
        extracted[header] = cleaned_content[:160].strip(" ;")

    if any(not extracted.get(header) for header in headers):
        return None

    text = "\n".join(f"{header} {extracted[header]}" for header in headers)
    if "bird silhouette, branch" in text.lower():
        return None

    if len(text) > 800:
        text = text[:800].strip()

    return text


def build_svg_input(prompt: str, plan_text=None) -> str:
    if USE_LAYOUT_PLAN and plan_text:
        return f"Prompt: {prompt}\nPlan:\n{plan_text}\nSVG:\n"
    return f"Prompt: {prompt}\nSVG:\n"


def build_revision_input(
    prompt: str,
    plan_text,
    current_svg: str,
    current_reason: str,
    gate_reason: str = "",
    strict_issues: str = "",
    hit_token_cap: bool = False,
) -> str:
    parts = [f"Prompt: {prompt}\n"]
    if plan_text:
        parts.extend(["Plan:\n", f"{plan_text}\n"])
    parts.append(f"Current issue: {current_reason}\n")
    if gate_reason:
        parts.append(f"Current gate reason: {gate_reason}\n")
    if strict_issues:
        parts.append(f"Strict contract issues: {strict_issues}\n")
    parts.append(f"Hit token cap: {'yes' if hit_token_cap else 'no'}\n")
    parts.extend(
        [
            "Current SVG:\n",
            f"{current_svg}\n",
            "Rewrite the SVG so it is valid, strict-contract compliant, concise, and more faithful to the prompt. Avoid scripts, animation, event handlers, external references, and remote URLs. Output SVG only.\n",
            "SVG:\n",
        ]
    )
    return "".join(parts)


def generate_text_batch(inputs_text, batch_size, max_new_tokens, do_sample=False):
    batch_outputs = []
    next_index = 0
    current_batch_size = max(1, batch_size)
    progress_bar = tqdm(total=len(inputs_text))

    while next_index < len(inputs_text):
        batch_inputs_text = inputs_text[next_index:next_index + current_batch_size]
        inputs = None
        outputs = None
        try:
            inputs = tokenizer(
                batch_inputs_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=do_sample,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_width = int(inputs["input_ids"].shape[1])
            generated_only = outputs[:, input_width:].detach().cpu()
            decoded_batch = tokenizer.batch_decode(generated_only, skip_special_tokens=True)
            eos_token_id = tokenizer.eos_token_id
            for raw_text, generated_ids in zip(decoded_batch, generated_only):
                generated_token_ids = generated_ids.tolist()
                if eos_token_id is not None and eos_token_id in generated_token_ids:
                    eos_index = generated_token_ids.index(eos_token_id)
                    effective_token_ids = generated_token_ids[:eos_index]
                    hit_token_cap = False
                else:
                    effective_token_ids = generated_token_ids
                    hit_token_cap = len(effective_token_ids) >= max_new_tokens
                generated_tokens = len(effective_token_ids)
                batch_outputs.append(
                    {
                        "raw_text": raw_text,
                        "generated_tokens": generated_tokens,
                        "hit_token_cap": hit_token_cap,
                    }
                )
            next_index += len(batch_inputs_text)
            progress_bar.update(len(batch_inputs_text))
        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM. Retrying with batch size {current_batch_size}.")
        finally:
            del inputs
            del outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    progress_bar.close()
    return batch_outputs


def build_candidate(prompt: str, generation_output: dict, candidate_source: str):
    candidate = clean_svg_output(generation_output.get("raw_text", ""))
    candidate["candidate_source"] = candidate_source
    candidate["generated_tokens"] = int(generation_output.get("generated_tokens", 0))
    candidate["hit_token_cap"] = as_bool(generation_output.get("hit_token_cap", False))
    return candidate


def candidate_sort_key(candidate: dict):
    return (
        REASON_RANK.get(str(candidate.get("reason", "")), -1),
        int(as_bool(candidate.get("strict_contract", False))),
        int(not as_bool(candidate.get("hit_token_cap", False))),
        min(len(str(candidate.get("final_svg", ""))), 8000),
    )


def choose_better_candidate(primary_candidate: dict, alternate_candidate=None):
    if alternate_candidate is None:
        return primary_candidate, False
    if candidate_sort_key(alternate_candidate) > candidate_sort_key(primary_candidate):
        return alternate_candidate, True
    return primary_candidate, False


def run_plan_pass(test_df, plan_batch_size=PLAN_BATCH_SIZE):
    plan_df = test_df[["id", "prompt"]].copy()
    if not USE_LAYOUT_PLAN:
        plan_df["plan_text"] = ""
        plan_df["plan_source"] = "disabled"
        plan_df["plan_raw_text"] = ""
        plan_df["plan_generated_tokens"] = 0
        plan_df["plan_hit_token_cap"] = False
        return reorder_columns(plan_df, PASS1_COLUMNS)

    plan_outputs = generate_text_batch(
        [build_plan_input(prompt) for prompt in plan_df["prompt"].tolist()],
        batch_size=plan_batch_size,
        max_new_tokens=PLAN_MAX_NEW_TOKENS,
        do_sample=False,
    )
    cleaned_plans = [clean_plan_output(output["raw_text"]) for output in plan_outputs]
    plan_df["plan_text"] = [
        cleaned if cleaned else build_fallback_plan(prompt)
        for prompt, cleaned in zip(plan_df["prompt"].tolist(), cleaned_plans)
    ]
    plan_df["plan_source"] = ["model" if cleaned else "fallback" for cleaned in cleaned_plans]
    plan_df["plan_raw_text"] = [output["raw_text"] for output in plan_outputs]
    plan_df["plan_generated_tokens"] = [int(output["generated_tokens"]) for output in plan_outputs]
    plan_df["plan_hit_token_cap"] = [as_bool(output["hit_token_cap"]) for output in plan_outputs]

    print("Plan source counts:")
    print(plan_df["plan_source"].value_counts())
    print(f"Plan token-cap hits: {int(plan_df['plan_hit_token_cap'].map(as_bool).sum())}")
    return reorder_columns(plan_df, PASS1_COLUMNS)


def run_svg_pass(pass1_df, svg_batch_size=SVG_BATCH_SIZE):
    working_df = pass1_df.copy()
    for column, default in {
        "plan_text": "",
        "plan_source": "",
        "plan_raw_text": "",
        "plan_generated_tokens": 0,
        "plan_hit_token_cap": False,
    }.items():
        if column not in working_df.columns:
            working_df[column] = default

    prompts = working_df["prompt"].astype(str).tolist()
    plan_texts = [as_text(value).strip() for value in working_df["plan_text"]]

    baseline_outputs = generate_text_batch(
        [build_svg_input(prompt, None) for prompt in prompts],
        batch_size=svg_batch_size,
        max_new_tokens=SVG_MAX_NEW_TOKENS,
        do_sample=False,
    )
    baseline_candidates = [
        build_candidate(prompt, output, "baseline")
        for prompt, output in zip(prompts, baseline_outputs)
    ]

    plan_candidates = [None] * len(working_df)
    if USE_LAYOUT_PLAN:
        plan_outputs = generate_text_batch(
            [build_svg_input(prompt, plan_text) for prompt, plan_text in zip(prompts, plan_texts)],
            batch_size=svg_batch_size,
            max_new_tokens=SVG_MAX_NEW_TOKENS,
            do_sample=False,
        )
        plan_candidates = [
            build_candidate(prompt, output, "plan")
            for prompt, output in zip(prompts, plan_outputs)
        ]

    selected_candidates = []
    used_plan_candidate = []
    debug_rows = []
    for index, row in working_df.iterrows():
        baseline_candidate = baseline_candidates[index]
        plan_candidate = plan_candidates[index]
        selected_candidate = baseline_candidate
        used_plan = False
        if USE_LAYOUT_PLAN and plan_candidate is not None:
            selected_candidate, used_plan = choose_better_candidate(plan_candidate, baseline_candidate)
            if selected_candidate is baseline_candidate:
                used_plan = False

        selected_candidates.append(selected_candidate)
        used_plan_candidate.append(used_plan)

        debug_row = {
            "id": row["id"],
            "prompt": row["prompt"],
            "plan_text": as_text(row.get("plan_text", "")),
            "plan_source": as_text(row.get("plan_source", "")),
            "plan_raw_text": as_text(row.get("plan_raw_text", "")),
            "plan_generated_tokens": int(row.get("plan_generated_tokens", 0)),
            "plan_hit_token_cap": as_bool(row.get("plan_hit_token_cap", False)),
            "used_plan_candidate": used_plan,
        }
        debug_row.update(candidate_base_row(selected_candidate))
        debug_row.update(candidate_to_prefixed_row("plan_candidate", plan_candidate))
        debug_row.update(candidate_to_prefixed_row("baseline_candidate", baseline_candidate))
        debug_rows.append(debug_row)

    pass2_debug_df = pd.DataFrame(debug_rows)
    pass2_submission_df = pd.DataFrame(
        {
            "id": working_df["id"].tolist(),
            "svg": [candidate["final_svg"] for candidate in selected_candidates],
        }
    )

    print("Baseline reason counts:")
    print(pd.Series([candidate["reason"] for candidate in baseline_candidates]).value_counts())
    if USE_LAYOUT_PLAN and any(candidate is not None for candidate in plan_candidates):
        print("Plan-conditioned reason counts:")
        print(pd.Series([candidate["reason"] for candidate in plan_candidates if candidate is not None]).value_counts())
    print("Selected pass 2 reason counts:")
    print(pass2_debug_df["reason"].value_counts())
    print(f"Pass 2 strict-contract pass rate: {float(pass2_debug_df['strict_contract'].map(as_bool).mean()):.4f}")

    preferred_columns = [
        "id",
        "prompt",
        "plan_text",
        "plan_source",
        "plan_raw_text",
        "plan_generated_tokens",
        "plan_hit_token_cap",
        "used_plan_candidate",
    ] + CANDIDATE_COLUMNS
    preferred_columns.extend(f"plan_candidate_{column}" for column in CANDIDATE_COLUMNS)
    preferred_columns.extend(f"baseline_candidate_{column}" for column in CANDIDATE_COLUMNS)
    pass2_debug_df = reorder_columns(pass2_debug_df, preferred_columns)
    return pass2_submission_df, pass2_debug_df


def run_revise_pass(pass1_df, pass2_debug_df, revise_batch_size=REVISE_BATCH_SIZE):
    pass1_by_id = pass1_df.drop_duplicates(subset=["id"]).set_index("id")
    working_df = pass2_debug_df.copy()
    for column in ("prompt", "plan_text", "plan_source", "plan_raw_text"):
        if column not in working_df.columns:
            working_df[column] = working_df["id"].map(pass1_by_id[column]) if column in pass1_by_id.columns else ""
    if "plan_generated_tokens" not in working_df.columns:
        working_df["plan_generated_tokens"] = working_df["id"].map(pass1_by_id["plan_generated_tokens"]) if "plan_generated_tokens" in pass1_by_id.columns else 0
    if "plan_hit_token_cap" not in working_df.columns:
        working_df["plan_hit_token_cap"] = working_df["id"].map(pass1_by_id["plan_hit_token_cap"]) if "plan_hit_token_cap" in pass1_by_id.columns else False

    pass2_candidates = [candidate_from_row(row) for _, row in working_df.iterrows()]

    revise_candidates_by_index = {}
    revision_attempted = [False] * len(working_df)
    revision_indices = [
        index
        for index, candidate in enumerate(pass2_candidates)
        if USE_REVISE_PASS and (
            candidate.get("reason") in REVISE_WEAK_REASONS
            or not as_bool(candidate.get("strict_contract", False))
            or as_bool(candidate.get("hit_token_cap", False))
        )
    ]

    if revision_indices:
        revise_inputs = []
        for index in revision_indices:
            row = working_df.iloc[index]
            candidate = pass2_candidates[index]
            revise_inputs.append(
                build_revision_input(
                    prompt=as_text(row["prompt"]),
                    plan_text=as_text(row.get("plan_text", "")).strip() or None,
                    current_svg=as_text(candidate.get("final_svg", "")),
                    current_reason=as_text(candidate.get("reason", "")),
                    gate_reason=as_text(candidate.get("gate_reason", "")),
                    strict_issues=as_text(candidate.get("strict_issues", "")),
                    hit_token_cap=as_bool(candidate.get("hit_token_cap", False)),
                )
            )

        revise_outputs = generate_text_batch(
            revise_inputs,
            batch_size=revise_batch_size,
            max_new_tokens=REVISE_MAX_NEW_TOKENS,
            do_sample=False,
        )

        for index, output in zip(revision_indices, revise_outputs):
            revision_attempted[index] = True
            revise_candidates_by_index[index] = build_candidate(
                as_text(working_df.iloc[index]["prompt"]),
                output,
                "revise",
            )

    final_candidates = []
    revision_used = []
    debug_rows = []
    for index, row in working_df.iterrows():
        pass2_candidate = pass2_candidates[index]
        revise_candidate = revise_candidates_by_index.get(index)
        selected_candidate, used_revision = choose_better_candidate(pass2_candidate, revise_candidate)
        final_candidates.append(selected_candidate)
        revision_used.append(used_revision)

        debug_row = {
            "id": row["id"],
            "prompt": as_text(row["prompt"]),
            "plan_text": as_text(row.get("plan_text", "")),
            "plan_source": as_text(row.get("plan_source", "")),
            "plan_raw_text": as_text(row.get("plan_raw_text", "")),
            "plan_generated_tokens": int(row.get("plan_generated_tokens", 0)),
            "plan_hit_token_cap": as_bool(row.get("plan_hit_token_cap", False)),
            "revision_attempted": revision_attempted[index],
            "revision_used": used_revision,
            "revised": used_revision,
            "pass2_reason": as_text(pass2_candidate.get("reason", "")),
            "pass2_candidate_source": as_text(pass2_candidate.get("candidate_source", "")),
        }
        debug_row.update(candidate_base_row(selected_candidate))
        debug_row.update(candidate_to_prefixed_row("pass2", pass2_candidate))
        debug_row.update(candidate_to_prefixed_row("revise", revise_candidate))
        debug_rows.append(debug_row)

    debug_df = pd.DataFrame(debug_rows)
    submission_df = pd.DataFrame(
        {
            "id": working_df["id"].tolist(),
            "svg": [candidate["final_svg"] for candidate in final_candidates],
        }
    )

    print(f"Revision attempts: {sum(revision_attempted)}")
    print(f"Revision replacements: {sum(revision_used)}")
    print("Final reason counts:")
    print(debug_df["reason"].value_counts())
    print(f"Final strict-contract pass rate: {float(debug_df['strict_contract'].map(as_bool).mean()):.4f}")

    preferred_columns = [
        "id",
        "prompt",
        "plan_text",
        "plan_source",
        "plan_raw_text",
        "plan_generated_tokens",
        "plan_hit_token_cap",
        "revision_attempted",
        "revision_used",
        "revised",
        "pass2_reason",
        "pass2_candidate_source",
    ] + CANDIDATE_COLUMNS
    preferred_columns.extend(f"pass2_{column}" for column in CANDIDATE_COLUMNS)
    preferred_columns.extend(f"revise_{column}" for column in CANDIDATE_COLUMNS)
    debug_df = reorder_columns(debug_df, preferred_columns)
    return submission_df, debug_df


## Pass 1: Layout Planning

This cell builds a compact plan for every prompt and saves the result immediately.

- Reads: `test.csv`, tokenizer, model
- Writes: `submission_pass1_plans.csv`
- Rerun-safe: Yes. It overwrites the saved pass-1 artifact.


In [ ]:
test_df = load_test_df(test_csv_path=TEST_CSV, limit=RUN_LIMIT)
pass1_df = run_plan_pass(test_df, plan_batch_size=PLAN_BATCH_SIZE)
save_dataframe(pass1_df, PASS1_PLANS_CSV, "pass 1 plans")
display(pass1_df.head())


## Pass 2: SVG Generation

This cell generates SVGs from the prompt plus plan when available, then saves the pass-2 submission and debug outputs.

- Reads: pass-1 plans, tokenizer, model, SVG helper functions
- Writes: `submission_pass2.csv`, `submission_pass2_debug.csv`
- Rerun-safe: Yes. It overwrites the saved pass-2 artifacts.


In [ ]:
if "pass1_df" not in globals():
    pass1_df = pd.read_csv(PASS1_PLANS_CSV, keep_default_na=False)

pass2_submission_df, pass2_debug_df = run_svg_pass(pass1_df, svg_batch_size=SVG_BATCH_SIZE)
save_dataframe(pass2_submission_df, PASS2_SUBMISSION_CSV, "pass 2 submission")
save_dataframe(pass2_debug_df, PASS2_DEBUG_CSV, "pass 2 debug")
display(pass2_debug_df.head())


## Pass 3: Targeted Revision And Final Build

This cell revises only weak pass-2 rows, then writes the final `submission.csv` and `submission_debug.csv`.

- Reads: pass-1 plans, pass-2 debug, tokenizer, model
- Writes: `submission.csv`, `submission_debug.csv`
- Rerun-safe: Yes. It overwrites the final saved output files.


In [ ]:
if "pass1_df" not in globals():
    pass1_df = pd.read_csv(PASS1_PLANS_CSV, keep_default_na=False)
if "pass2_debug_df" not in globals():
    pass2_debug_df = pd.read_csv(PASS2_DEBUG_CSV, keep_default_na=False)

submission_df, debug_df = run_revise_pass(pass1_df, pass2_debug_df, revise_batch_size=REVISE_BATCH_SIZE)
save_dataframe(submission_df, SUBMISSION_CSV, "final submission")
save_dataframe(debug_df, DEBUG_CSV, "final debug")


## Final Review

This next cell reads the saved output files back from Drive and previews the final submission and debug dataframes.

- Reads: `submission.csv`, `submission_debug.csv`
- Writes: Nothing
- Rerun-safe: Yes. Read-only review cell.


In [ ]:
saved_submission_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
saved_debug_df = pd.read_csv(DEBUG_CSV, keep_default_na=False)

print(f"Saved submission rows: {len(saved_submission_df)}")
print(f"Saved debug rows: {len(saved_debug_df)}")
if "strict_contract" in saved_debug_df.columns:
    strict_mask = saved_debug_df["strict_contract"].map(as_bool)
    print(f"Saved strict-contract pass rate: {float(strict_mask.mean()):.4f}")
    print(f"Saved rows with strict issues: {int((~strict_mask).sum())}")
print(saved_debug_df["reason"].value_counts())
display(saved_submission_df.head())
display(saved_debug_df.head())
